# Code for Volatility Targeting: An Application of the Volatility Forecasting 

## Does forecast quality translate into better portfolio risk management?

#### Basic Idea: Instead of holding a fixed position size, you scale your position every day so that your expected portfolio volatility equals a fixed target level.

$$ w_t = \frac{\sigma_{target}}{\hat{\sigma}} $$
where:
* $w_t$ is portfolio weight/ levarage at time t
* $\sigma_{target} $ is the desired daily volatility
* $ \hat{\sigma}$ is forecasted daily volatility

If forecasted volatility is high, reduce exposure

If forecasted volatility is low, increase exposure

I am going to use a leverage cap: 
$$ 0 \leq w_t \leq 2 $$
and I will use a 10% annualized volatility target as an illustrative fixed target


Steps:
1. Load data and model forecasts
2. Build target-vol weights
3. Compute strategy returns
4. Compare against buy-and-hold
5. Plot equity curves, rolling vol, drawdown
6. Report Sharpe/vol/drawdown/turnover

In [ ]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go


sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset
from baselines import make_baseline_forecasts
from metrics import qlike, mse, score, score_by_regime, score_by_ticker
from garch import add_garch_refit_recurse, add_gjr_garch_forecast, plot_vol_compare
from har_rv_ensemble import add_har_forecasts

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}

DEV_START = "2000-01-10"
EVAL_START = "2005-01-01"


In [26]:
df_all = pd.read_csv("volatility_forecasts.csv")

# Convert date strings to pandas datetime objects
df_all["date"] = pd.to_datetime(df_all["date"])
df_all.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,garch5_var,gjr1_var,gjr5_var,rv_d,rv_w,rv_m,har1_var,har5_var,ens1_var,ens5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,...,NaN,NaN,NaN,5.704863,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,...,NaN,NaN,NaN,1.448977,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,...,NaN,NaN,NaN,0.390316,NaN,NaN,NaN,NaN,NaN,NaN


-----------------------------------------------------------------------
# SECTION A: I will take SPY as the only traded asset for this section.
-----------------------------------------------------------------------

### Step 1: Isolate and prepare SPY data.
The returns ret calculated in notebook 01_data_pipeline were log returns: $$ 100 \log(P_t/P_{t-1}) $$

For portfolio returns, I will convert them to simple decimal return

In [27]:
spy = ( df_all[df_all["ticker"] == "SPY"]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
        )

# ret = 1 means a 1% log return.
# ret / 100 converts it to decimal log return.
# exp(log return) - 1 gives the corresponding simple return.
#The numpy.expm1 function calculates \(e^x - 1\) element-wise for all values in an input array.
# Its primary advantage is providing significantly higher numerical precision than using np.exp(x) - 1

spy["simple_ret"] = np.expm1(spy["ret"] / 100.0)
spy.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,gjr1_var,gjr5_var,rv_d,rv_w,rv_m,har1_var,har5_var,ens1_var,ens5_var,simple_ret
0,2000-01-10,SPY,146.25000,146.90625,145.03125,146.25000,91.641846,5741700,0.342420,0.117251,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003430
1,2000-01-11,SPY,145.81250,146.09375,143.50000,144.50000,90.545334,7503700,-1.203735,1.448977,...,NaN,NaN,1.448977,NaN,NaN,NaN,NaN,NaN,NaN,-0.011965
2,2000-01-12,SPY,144.59375,144.59375,142.87500,143.06250,89.644539,6907700,-0.999837,0.999674,...,NaN,NaN,0.999674,NaN,NaN,NaN,NaN,NaN,NaN,-0.009949
3,2000-01-13,SPY,144.46875,145.75000,143.28125,145.00000,90.858620,5158300,1.345239,1.809667,...,NaN,NaN,1.809667,NaN,NaN,NaN,NaN,NaN,NaN,0.013543
4,2000-01-14,SPY,146.53125,147.46875,145.96875,146.96875,92.092293,7437300,1.348659,1.818881,...,NaN,NaN,1.818881,NaN,NaN,NaN,NaN,NaN,NaN,0.013578


### Step 2: Convert variance forecasts into annualized volatility

Since the variance forecasts produced by all the models in the previous notebooks are in units (percentage return)^2, I will convert them into annualized daily volatility

In [28]:
# For GJR-GARCH
# sqrt(gjr1_var) gives daily volatility in percentage units.
# Divide by 100 to convert percentage -> decimal.
# Multiply by sqrt(252) to annualize.

spy["gjr_ann_vol"] = (
    np.sqrt(spy["gjr1_var"])
    / 100.0
    * np.sqrt(252)
)

# For EWMA
spy["ewma_ann_vol"] = (
    np.sqrt(spy["ewma1_var"])
    / 100.0
    * np.sqrt(252)
)

# For HAR: Since it predicts total five-day variance, first I will compute average expected daily variance by dividing by 5 and then convert into annualized daily volatility
spy["har_ann_vol"] = (
    np.sqrt(spy["har5_var"] / 5.0)
    / 100.0
    * np.sqrt(252)
)

### Step 3: Construct volatility-targeting weights

In [29]:
TARGET_VOL = 0.10       # 10% annualized target
MAX_LEVERAGE = 2.0      # maximum exposure = 200%

# For GJR-GARCH
# Raw volatility-targeting weight
spy["gjr_weight_signal"] = TARGET_VOL / spy["gjr_ann_vol"]

# Prevent extreme leverage during very low-volatility periods
spy["gjr_weight_signal"] = spy["gjr_weight_signal"].clip(
    lower=0.0,
    upper=MAX_LEVERAGE
)


# For EWMA
spy["ewma_weight_signal"] = (
    TARGET_VOL / spy["ewma_ann_vol"]
).clip(
    lower=0.0,
    upper=MAX_LEVERAGE
)

#For HAR
spy["har_weight_signal"] = (
    TARGET_VOL / spy["har_ann_vol"]
).clip(
    lower=0.0,
    upper=MAX_LEVERAGE
)

#### Since at time $t$, the forecasts predict future volatility, therefore the weight computed at $t$ is applied to return at $t+1$

In [30]:
# Shift weights forward one trading day.
# Forecast at t -> position held during t+1.

spy["gjr_weight"] = spy["gjr_weight_signal"].shift(1)
spy["ewma_weight"] = spy["ewma_weight_signal"].shift(1)
spy["har_weight"] = spy["har_weight_signal"].shift(1)

In [31]:
# Static constant-risk benchmark

# DEV_START = "2000-01-10"
# EVAL_START = "2005-01-01"


# Use ONLY the development period to estimate SPY volatility.
# We do not use evaluation-period information because that
# would introduce look-ahead bias.
dev_spy = spy[
    (spy["date"] >= pd.Timestamp(DEV_START)) &
    (spy["date"] < pd.Timestamp(EVAL_START))
].copy()


# Estimate annualized SPY volatility during the development period.
dev_spy_ann_vol = (
    dev_spy["simple_ret"].std()
    * np.sqrt(252)
)


# Choose ONE fixed exposure that would have targeted 10% volatility
# based on the information available before the test period.
static_weight = TARGET_VOL / dev_spy_ann_vol


# Apply the same leverage constraint as the dynamic strategies.
static_weight = np.clip(
    static_weight,
    0.0,
    MAX_LEVERAGE
)

print("Development-period SPY annualized volatility:",
      dev_spy_ann_vol)

print("Static SPY exposure:",
      static_weight)

Development-period SPY annualized volatility: 0.2053948904011567
Static SPY exposure: 0.4868670287011037


### Step 4: Calculate strategy returns (without any transaction costs)

In [32]:
# Simple Buy-and-hold SPY
spy["buy_hold_ret"] = spy["simple_ret"]

# Volatility-targeted strategies
spy["gjr_vt_ret_gross"] = (
    spy["gjr_weight"] * spy["simple_ret"]
)

spy["ewma_vt_ret_gross"] = (
    spy["ewma_weight"] * spy["simple_ret"]
)

spy["har_vt_ret_gross"] = (
    spy["har_weight"] * spy["simple_ret"]
)

### Step 5: Include transaction costs
* Basis Point(BP): One basis point is equal to one-hundredth of a percentage point (0.01% or 0.0001 in decimal form).

* Turnover: It measure the absolute change in the portfolio's assest allocations from one day (or period) to the next. One "unit" of turnover means you traded an amout equal to 100% of your total portfolio value.

In volatilty targeting strategy, the goal is to keep the portfolio's risk constant leading to selling of assets to lower exposure if market volatility spikes, and buying of assets to scale the risk back up to the target when market volatility drops. Every single one of these buying and selling actions creates turnover.

I will use 2 basis points per unit of turnover in my calculations. This means the every time I rebalance the portfolio, I pay a trading cost equal to 0.02% of the total dollar volume shifted.

Hence,
$$ cost_t = | w_t - w_{t-1} | $$

In [33]:
COST_BPS = 2
# Convert basis points into decimal return units
cost_rate = COST_BPS / 10000.0


# For GJR-GARCH
spy["gjr_turnover"] = spy["gjr_weight"].diff().abs() # Absolute change in portfolio exposure
spy["gjr_cost"] = cost_rate * spy["gjr_turnover"] # Trading cost
spy["gjr_vt_ret"] = ( spy["gjr_vt_ret_gross"] - spy["gjr_cost"] ) # Net strategy return


# For EWMA
spy["ewma_turnover"] = spy["ewma_weight"].diff().abs()
spy["ewma_cost"] = cost_rate * spy["ewma_turnover"]
spy["ewma_vt_ret"] = ( spy["ewma_vt_ret_gross"] - spy["ewma_cost"] )


# For HAR
spy["har_turnover"] = spy["har_weight"].diff().abs()
spy["har_cost"] = cost_rate * spy["har_turnover"]
spy["har_vt_ret"] = ( spy["har_vt_ret_gross"] - spy["har_cost"] )


### Step 6: Performance

In [34]:
def strategy_metrics( returns, weights=None, trading_days=252):

    """
    Calculate basic strategy performance statistics.
    Returns are decimal simple returns.
    """

    # Remove missing observations
    r = returns.dropna()
    n = len(r)


    # ---------------------------------------
    # Geometric annualized return
    # ---------------------------------------
    total_growth = (1 + r).prod()
    ann_return = total_growth ** (trading_days / n) - 1


    # ---------------------------------------
    # Annualized volatility
    # ---------------------------------------
    ann_vol = r.std() * np.sqrt(trading_days)


    # ---------------------------------------
    # Sharpe ratio
    # Assume zero risk-free rate for simplicity.
    # ---------------------------------------
    sharpe = ( r.mean() / r.std() * np.sqrt(trading_days) )


    # ---------------------------------------
    # Wealth curve
    # ---------------------------------------
    wealth = (1 + r).cumprod()

    # Previous maximum wealth
    running_max = wealth.cummax()

    # Percentage drawdown
    drawdown = wealth / running_max - 1

    # Worst drawdown
    max_drawdown = drawdown.min()


    # ---------------------------------------
    # Worst daily return
    # ---------------------------------------
    worst_day = r.min()


    # ---------------------------------------
    # Average exposure and turnover
    # ---------------------------------------
    avg_weight = np.nan
    ann_turnover = np.nan

    if weights is not None:

        w = weights.loc[r.index]

        avg_weight = w.mean()

        # Average daily turnover × 252
        ann_turnover = (
            w.diff().abs().mean()
            * trading_days
        )

    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "worst_day": worst_day,
        "avg_weight": avg_weight,
        "ann_turnover": ann_turnover
    }

In [35]:
# Use the final OOS period only.

# EVAL_START = "2005-01-01"
# Subset of spy for the out-of-sample backtest
bt = spy[ spy["date"] >= pd.Timestamp(EVAL_START) ].copy()

# The static strategy uses exactly the same exposure every day.
bt["static_weight"] = static_weight


# Static risk-scaled SPY return.
#
# Unlike volatility targeting, this weight does NOT react
# to changing market volatility.
bt["static_risk_ret"] = (
    bt["static_weight"]
    * bt["simple_ret"]
)


results = []

# Buy and hold
m = strategy_metrics(
    bt["buy_hold_ret"]
)

m["strategy"] = "Buy & Hold"
results.append(m)

# Static constant-risk SPY benchmark
m = strategy_metrics(
    bt["static_risk_ret"],
    bt["static_weight"]
)

m["strategy"] = "Static Risk"
results.append(m)


# EWMA volatility targeting
m = strategy_metrics(
    bt["ewma_vt_ret"],
    bt["ewma_weight"]
)

m["strategy"] = "EWMA VT"
results.append(m)


# GJR volatility targeting
m = strategy_metrics(
    bt["gjr_vt_ret"],
    bt["gjr_weight"]
)

m["strategy"] = "GJR VT"
results.append(m)


# HAR volatility targeting
m = strategy_metrics(
    bt["har_vt_ret"],
    bt["har_weight"]
)

m["strategy"] = "HAR VT"
results.append(m)


performance = pd.DataFrame(results)

performance = performance[
    [
        "strategy",
        "ann_return",
        "ann_vol",
        "sharpe",
        "max_drawdown",
        "worst_day",
        "avg_weight",
        "ann_turnover"
    ]
]

performance

,strategy,ann_return,ann_vol,sharpe,max_drawdown,worst_day,avg_weight,ann_turnover
0,Buy & Hold,0.109077,0.189144,0.642093,-0.551894,-0.109424,NaN,NaN
1,Static Risk,0.056418,0.092088,0.642093,-0.305924,-0.053275,0.486867,0.000000
2,EWMA VT,0.075886,0.106957,0.737623,-0.237245,-0.056793,0.787889,6.509415
3,GJR VT,0.070897,0.100409,0.732608,-0.254805,-0.050415,0.777052,9.698529
4,HAR VT,0.075709,0.114277,0.696046,-0.280021,-0.072576,0.860060,27.329532


### Step 7: Cumulative wealth plot

In [36]:
bt["Buy & Hold"] = (
    1 + bt["buy_hold_ret"].fillna(0)
).cumprod()

bt["Static Risk"] = (
    1 + bt["static_risk_ret"].fillna(0)
).cumprod()

bt["EWMA VT"] = (
    1 + bt["ewma_vt_ret"].fillna(0)
).cumprod()

bt["GJR VT"] = (
    1 + bt["gjr_vt_ret"].fillna(0)
).cumprod()

bt["HAR VT"] = (
    1 + bt["har_vt_ret"].fillna(0)
).cumprod()

In [37]:
fig = go.Figure()

for col in [
    "Buy & Hold",
    "Static Risk",
    "EWMA VT",
    "GJR VT",
    "HAR VT"
]:
    fig.add_trace(
        go.Scatter(
            x=bt["date"],
            y=bt[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title="SPY Volatility Targeting: Cumulative Wealth",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    template="plotly_white"
)

fig.show()

### Step 8: Did we actually hit the volatility target?

In [38]:
WINDOW = 63   # approximately three months

bt["buy_hold_realized_vol"] = (
    bt["buy_hold_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

bt["static_realized_vol"] = (
    bt["static_risk_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

bt["gjr_realized_vol"] = (
    bt["gjr_vt_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

bt["ewma_realized_vol"] = (
    bt["ewma_vt_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

bt["har_realized_vol"] = (
    bt["har_vt_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

In [39]:
fig = go.Figure()

for col, name in [
    ("buy_hold_realized_vol", "Buy & Hold"),
    ("static_realized_vol", "Static Risk"),
    ("ewma_realized_vol", "EWMA VT"),
    ("gjr_realized_vol", "GJR VT"),
    ("har_realized_vol", "HAR VT")
]:

    fig.add_trace(
        go.Scatter(
            x=bt["date"],
            y=bt[col],
            mode="lines",
            name=name
        )
    )

# Target volatility
fig.add_hline(
    y=TARGET_VOL,
    line_dash="dash",
    annotation_text="10% Target"
)

fig.update_layout(
    title="Rolling Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    template="plotly_white"
)

fig.show()

-----------------------------------------------------------------------
# SECTION B: I will look at portfolio-level volatility targeting by constructing a fixed weight portfolio using SPY and JPM in this section.
-----------------------------------------------------------------------

### Step 1: Construct a fixed-weight portfolio
 The fixed-weight portfolio return at time $t$ given by
$$ r_{p,t} = w_{\scriptscriptstyle SPY} r_{{\scriptscriptstyle SPY},t} + w_{\scriptscriptstyle JPM} r_{{\scriptscriptstyle JPM},t} $$
I will take a 50-50 SPY-JPM portfolio, i.e.,
$$ w_{\scriptscriptstyle SPY} = w_{\scriptscriptstyle JPM} = 0.5 $$


In [40]:
# Put SPY and JPM simple returns side-by-side
portfolio = (
    df_all[df_all["ticker"].isin(["SPY", "JPM"])]
    .pivot(index="date", columns="ticker", values="ret")
    .dropna()
)

# Convert percentage log returns to decimal simple returns
portfolio["SPY_ret"] = np.expm1(portfolio["SPY"] / 100)
portfolio["JPM_ret"] = np.expm1(portfolio["JPM"] / 100)

# Simple 50/50 portfolio return
SPY_WEIGHT = 0.5
JPM_WEIGHT = 0.5

portfolio["portfolio_ret"] = (
    SPY_WEIGHT * portfolio["SPY_ret"]
    + JPM_WEIGHT * portfolio["JPM_ret"]
)

### Step 2: Add model forecast and rolling correlation
Note: In this section, I am considering the volatility forecast given by GJR-GARCH only and not EWMA & HAR as it has already been established that GJR is the preferred 1-day horizon model

In [41]:
# Put SPY and JPM 1-day GJR variance forecasts side-by-side
gjr_wide = (
    df_all[df_all["ticker"].isin(["SPY", "JPM"])]
    .pivot(index="date", columns="ticker", values="gjr1_var")
)

# Make sure the index is datetime
portfolio.index = pd.to_datetime(portfolio.index)
gjr_wide.index = pd.to_datetime(gjr_wide.index)

# Add individual GJR variance forecasts to the portfolio dataframe
portfolio["SPY_gjr_var"] = gjr_wide["SPY"]
portfolio["JPM_gjr_var"] = gjr_wide["JPM"]


# Estimate SPY-JPM correlation using the most recent 60 trading days
CORR_WINDOW = 60

portfolio["rolling_corr"] = (
    portfolio["SPY_ret"]
    .rolling(CORR_WINDOW, min_periods=20)
    .corr(portfolio["JPM_ret"])
)

### Step 3: Forecast portfolio variance and convert to annualized volatility
The portfolio variance is given by
$$ \sigma_p^2 = w_{\scriptscriptstyle SPY}^2 \sigma_{\scriptscriptstyle SPY}^2 + w_{\scriptscriptstyle JPM}^2 \sigma_{\scriptscriptstyle JPM}^2 + 2 w_{\scriptscriptstyle SPY} w_{\scriptscriptstyle JPM} Cov(SPY,JPM) $$
where covariance $ Cov(SPY, JPM) $ is given by
$$ Cov(SPY, JPM) = \rho_{\scriptscriptstyle (SPY,JPM)} \sigma_{\scriptscriptstyle SPY} \sigma_{\scriptscriptstyle JPM} $$



In [42]:

# Forecast covariance between SPY and JPM
portfolio["forecast_cov"] = (
    portfolio["rolling_corr"]
    * np.sqrt(portfolio["SPY_gjr_var"])
    * np.sqrt(portfolio["JPM_gjr_var"])
)


# Forecast variance of the 50/50 portfolio
portfolio["gjr_portfolio_var"] = (
    (SPY_WEIGHT ** 2) * portfolio["SPY_gjr_var"]
    +
    (JPM_WEIGHT ** 2) * portfolio["JPM_gjr_var"]
    +
    2* SPY_WEIGHT * JPM_WEIGHT * portfolio["forecast_cov"]
)

# Convert daily portfolio variance forecast to annualized decimal volatility
portfolio["gjr_ann_vol"] = (
    np.sqrt(portfolio["gjr_portfolio_var"])
    / 100.0
    * np.sqrt(252)
)

### Step 4: Construct dynamic GJR-GARCH volatility weights and calculate volatility-targeted net portfolio returns

In [43]:
# Desired portfolio exposure based on forecast volatility
portfolio["gjr_weight_signal"] = (
    TARGET_VOL / portfolio["gjr_ann_vol"]
)


# Prevent excessive leverage
portfolio["gjr_weight_signal"] = (
    portfolio["gjr_weight_signal"]
    .clip(lower=0.0, upper=MAX_LEVERAGE)
)


# Forecast made at t is used for the position held during t+1
portfolio["gjr_weight"] = (
    portfolio["gjr_weight_signal"].shift(1)
)

In [44]:
# Gross strategy return
portfolio["gjr_vt_ret_gross"] = (
    portfolio["gjr_weight"]
    * portfolio["portfolio_ret"]
)


# Turnover caused by changing total portfolio exposure
portfolio["gjr_turnover"] = (
    portfolio["gjr_weight"].diff().abs()
)


# Transaction cost same as section A with 
# COST_BPS = 2 ==> cost_rate = COST_BPS / 10000.0
portfolio["gjr_cost"] = (
    cost_rate * portfolio["gjr_turnover"]
)


# Net portfolio return after transaction costs
portfolio["gjr_vt_ret"] = (
    portfolio["gjr_vt_ret_gross"]
    - portfolio["gjr_cost"]
)

### Step 5: Calculate static-risk portfolio benchmark

In [45]:
# Development-period data only
# As before DEV_START = "2000-01-10" and EVAL_START = "2007-01-01"
portfolio_dev = portfolio[
    (portfolio.index >= pd.Timestamp(DEV_START)) &
    (portfolio.index < pd.Timestamp(EVAL_START))
].copy()


# Historical annualized volatility of the unscaled 50/50 portfolio
portfolio_dev_ann_vol = (
    portfolio_dev["portfolio_ret"].std()
    * np.sqrt(252)
)


# Choose one fixed exposure before the evaluation period begins
portfolio_static_weight = (
    TARGET_VOL / portfolio_dev_ann_vol
)


# Apply the same leverage cap used for dynamic strategies
portfolio_static_weight = np.clip(
    portfolio_static_weight,
    0.0,
    MAX_LEVERAGE
)


print("Development-period portfolio volatility:",
      portfolio_dev_ann_vol)

print("Static portfolio weight:",
      portfolio_static_weight)

Development-period portfolio volatility: 0.2892440611554477
Static portfolio weight: 0.34572879249630384


### Step 6: Create the portfolio backtest dataframe and evaluate performance

In [46]:
# Keep only the out-of-sample evaluation period
portfolio_bt = portfolio[
    portfolio.index >= pd.Timestamp(EVAL_START)
].copy()


# Unscaled 50/50 portfolio
portfolio_bt["buy_hold_ret"] = (
    portfolio_bt["portfolio_ret"]
)


# Static constant-risk portfolio
portfolio_bt["static_weight"] = (
    portfolio_static_weight
)

portfolio_bt["static_risk_ret"] = (
    portfolio_bt["static_weight"]
    * portfolio_bt["portfolio_ret"]
)

In [47]:
portfolio_results = []


# 50/50 Buy & Hold
m = strategy_metrics(
    portfolio_bt["buy_hold_ret"]
)

m["strategy"] = "50/50 Buy & Hold"
portfolio_results.append(m)


# Static risk-scaled 50/50 portfolio
m = strategy_metrics(
    portfolio_bt["static_risk_ret"],
    portfolio_bt["static_weight"]
)

m["strategy"] = "50/50 Static Risk"
portfolio_results.append(m)


# Dynamic GJR volatility targeting
m = strategy_metrics(
    portfolio_bt["gjr_vt_ret"],
    portfolio_bt["gjr_weight"]
)

m["strategy"] = "50/50 GJR VT"
portfolio_results.append(m)


# Create final comparison table
portfolio_performance = pd.DataFrame(portfolio_results)

portfolio_performance = portfolio_performance[
    [
        "strategy",
        "ann_return",
        "ann_vol",
        "sharpe",
        "max_drawdown",
        "worst_day",
        "avg_weight",
        "ann_turnover"
    ]
]

portfolio_performance

,strategy,ann_return,ann_vol,sharpe,max_drawdown,worst_day,avg_weight,ann_turnover
0,50/50 Buy & Hold,0.132416,0.254202,0.616112,-0.585159,-0.131781,NaN,NaN
1,50/50 Static Risk,0.051571,0.087885,0.616112,-0.221663,-0.045561,0.345729,0.000000
2,50/50 GJR VT,0.075136,0.099927,0.775098,-0.212629,-0.040767,0.604463,5.767923


### Step 7: Cumulative wealth and Rolling realized volatility plots

In [48]:
# Cumulative wealth for all three strategies, buy-hold, static risk, and GJR-GARCH volatility targeting

portfolio_bt["50/50 Buy & Hold"] = (
    1 + portfolio_bt["buy_hold_ret"].fillna(0)
).cumprod()

portfolio_bt["50/50 Static Risk"] = (
    1 + portfolio_bt["static_risk_ret"].fillna(0)
).cumprod()

portfolio_bt["50/50 GJR VT"] = (
    1 + portfolio_bt["gjr_vt_ret"].fillna(0)
).cumprod()

In [49]:
#Plot
fig = go.Figure()

for col in [
    "50/50 Buy & Hold",
    "50/50 Static Risk",
    "50/50 GJR VT"
]:
    fig.add_trace(
        go.Scatter(
            x=portfolio_bt.index,
            y=portfolio_bt[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title="50/50 SPY-JPM Portfolio: Cumulative Wealth",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    template="plotly_white"
)

fig.show()

In [50]:
# Rolling annualized volatility of each portfolio strategy
portfolio_bt["buy_hold_realized_vol"] = (
    portfolio_bt["buy_hold_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

portfolio_bt["static_realized_vol"] = (
    portfolio_bt["static_risk_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

portfolio_bt["gjr_realized_vol"] = (
    portfolio_bt["gjr_vt_ret"]
    .rolling(WINDOW)
    .std()
    * np.sqrt(252)
)

In [51]:
fig = go.Figure()

for col, name in [
    ("buy_hold_realized_vol", "50/50 Buy & Hold"),
    ("static_realized_vol", "50/50 Static Risk"),
    ("gjr_realized_vol", "50/50 GJR VT")
]:
    fig.add_trace(
        go.Scatter(
            x=portfolio_bt.index,
            y=portfolio_bt[col],
            mode="lines",
            name=name
        )
    )


# Target volatility
fig.add_hline(
    y=TARGET_VOL,
    line_dash="dash",
    annotation_text="10% Target"
)


fig.update_layout(
    title="50/50 SPY-JPM Portfolio: Rolling Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    template="plotly_white"
)

fig.show()